# 08. Сравнение всех экспериментов

Собирает результаты разных ноутбуков в одну таблицу и считает парные сравнения с базовой линией.

## Инструкция

1. В каждом обучающем ноутбуке нажмите `Save Version`, чтобы его выходные файлы сохранились.
2. Здесь через `Add Input` → `Notebook Output` подключите выходы нужных ноутбуков.
3. В ячейке «Пути» перечислите папки с результатами обучения (`experiments/freq`) и с изображениями (`sr`).
4. Запустите все ячейки. GPU не нужен, кроме пакета `lpips` (на CPU он работает медленнее).

**Как читать таблицу:** главная колонка — `hf_err` (ниже — лучше; выше 1 — детали выдуманы). Вывод «значимо»
означает, что 95 % интервал разности с базовой линией не содержит нуля.

In [ ]:
import os, glob
CODE_SRC = "/kaggle/input/pisa-sr-main/PiSA-SR-main"
VAL_HQ   = "/kaggle/input/datasets/vende14/hq-lq-netherlands-val/hq_images"
# Папки experiments/freq из выходов обучающих ноутбуков:
RESULT_ROOTS = sorted(glob.glob("/kaggle/input/*/experiments/freq"))
# Папки с изображениями (sr/<эксперимент>/lsem_1.0):
SR_ROOTS = sorted(glob.glob("/kaggle/input/*/sr"))
BASELINE = "E01"       # с чем сравнивать
print(RESULT_ROOTS, SR_ROOTS, sep="\n")

In [ ]:
import shutil
WORK = "/kaggle/working/PiSA-SR-main"
if not os.path.exists(WORK):
    shutil.copytree(CODE_SRC, WORK)
os.chdir(WORK)
!pip install -q lpips

## Итоговые точки кривых обучения

In [ ]:
import pandas as pd
rows = []
for root in RESULT_ROOTS:
    for d in sorted(glob.glob(os.path.join(root, "E*"))):
        cs = [pd.read_csv(f) for f in sorted(glob.glob(os.path.join(d, "curve_*.csv")))]
        if not cs: continue
        c = pd.concat(cs).sort_values("global_step")
        last = c[c.global_step >= c.global_step.max() - 1000]      # среднее по последним 1000 шагам (3 точки при EVAL_EVERY = 500)
        rows.append({"эксперимент": os.path.basename(d), "шаг": int(c.global_step.max()),
                     **last[["lpips_val", "vol_ratio", "hf_frc", "hf_err", "hf_energy", "eff_gsd_cm"]].mean().round(4).to_dict()})
table = pd.DataFrame(rows).sort_values("hf_err"); table

## Парные сравнения с базовой линией

In [ ]:
sr = {}
for root in SR_ROOTS:
    for d in sorted(glob.glob(os.path.join(root, "E*", "lsem_1.0"))):
        sr[os.path.basename(os.path.dirname(d))] = d
base = next((v for k, v in sr.items() if k.startswith(BASELINE)), None)
assert base, f"нет изображений базовой линии {BASELINE}"
for name, d in sr.items():
    if d != base:
        print("\n" + "=" * 20, name, "против", BASELINE)
        !python freq_exp/eval_bands.py --sr_dir {d} --sr_dir_b {base} --hq_dir {VAL_HQ} --n_boot 1000 | tail -12

## Кривые ошибки в высокочастотных полосах

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for root in RESULT_ROOTS:
    for d in sorted(glob.glob(os.path.join(root, "E*"))):
        cs = [pd.read_csv(f) for f in sorted(glob.glob(os.path.join(d, "curve_*.csv")))]
        if not cs: continue
        c = pd.concat(cs).sort_values("global_step"); c = c[c.stage == "sem"]
        ax[0].plot(c.global_step, c.hf_err, label=os.path.basename(d)[:28])
        ax[1].plot(c.global_step, c.lpips_val)
ax[0].axhline(1, ls=":", c="k"); ax[0].set_ylabel("ошибка в ВЧ-полосах, ниже — лучше"); ax[1].set_ylabel("LPIPS")
for a in ax: a.set_xlabel("шаг"); a.grid(alpha=.3)
ax[0].legend(fontsize=7); plt.tight_layout()